# train alphapeptdeep

In [12]:
import os
dfs = []
frag_inten_dfs = []            

from alphabase.spectral_library.reader import LibraryReaderBase
_lib = LibraryReaderBase()
base_path = r"D:\DACN\dbl_alphapepdeep\_lt\251128\251128"

with open(r"D:\DACN\dbl_alphapepdeep\_lt\251128\_index.tsv", "r") as file:
    for line in file:
        file_name = line.rstrip()  
        full_path = os.path.join(base_path, file_name)  
        a = _lib.import_file(full_path) 
        dfs.append(a)
        frag_inten_dfs.append(_lib.fragment_intensity_df)

from alphabase.peptide.fragment import concat_precursor_fragment_dataframes

psm_df, frag_df = concat_precursor_fragment_dataframes(dfs, frag_inten_dfs)

# fine tune
from peptdeep.pretrained_models import ModelManager
model_interface= ModelManager(device='cpu') 

supported_fragment_types = model_interface.model.supported_charged_frag_types
print(f"Fragment types supported by the model: {supported_fragment_types}")

psm_df['nce'] = 30.0
psm_df['instrument'] = "QE"
psm_df['precursor_idx'] = psm_df.index 

model_interface.train(precursor_df=psm_df, fragment_intensity_df=frag_df.loc[:, supported_fragment_types], epoch=30, verbose=1)

# evaluation
import pandas as pd
import numpy as np
def calculate_similarity(precursor_df_a, precursor_df_b, intensity_df_a, intensity_df_b):
    _a_df = precursor_df_a[['precursor_idx', 'frag_start_idx', 'frag_stop_idx']].copy()
    _b_df = precursor_df_b[['precursor_idx', 'frag_start_idx', 'frag_stop_idx']].copy()
    _merged_df = pd.merge(_a_df, _b_df, on='precursor_idx', suffixes=('_a', '_b'))
    # keep only first precursor
    _merged_df = _merged_df.drop_duplicates(subset='precursor_idx', keep='first')
    similarity_list = []
    for i, (start_a, stop_a, start_b, stop_b) in enumerate(zip(_merged_df['frag_start_idx_a'], _merged_df['frag_stop_idx_a'], _merged_df['frag_start_idx_b'], _merged_df['frag_stop_idx_b'])):
        observed_intensity = intensity_df_a.loc[start_a:stop_a, :].values.flatten()
        predicted_intensity = intensity_df_b.loc[start_b:stop_b, :].values.flatten()
        similarity = np.dot(observed_intensity, predicted_intensity) / ((np.linalg.norm(observed_intensity) * np.linalg.norm(predicted_intensity)) + 1e-6)
        similarity_list.append({'similarity': similarity, 'index': i, 'precursor_idx': _merged_df.iloc[i]['precursor_idx']})
    return pd.DataFrame(similarity_list)


print(f"Prediction after training on fragment types: {supported_fragment_types}") 
prediction_precursor_df = psm_df.copy()
prediction_precursor_df.drop(columns=["frag_start_idx", "frag_stop_idx"], inplace=True)
predictions = model_interface.predict(prediction_precursor_df) # predict wil set the frag_start_idx and frag_stop_idx inplace
similarity_df = calculate_similarity(psm_df, prediction_precursor_df, frag_df.loc[:, supported_fragment_types], predictions)
print(f"Median similarity: {similarity_df['similarity'].median()}")

# train from begining
# TP testing: requested_fragment_types not working yet
requested_fragment_types = supported_fragment_types

model_interface = pDeepModel(charged_frag_types=requested_fragment_types) # creating a model from scratch
#check pdeep.build
#check pdeep.grid_nce_search
model_interface.train(precursor_df=psm_df, fragment_intensity_df=frag_df.loc[:,supported_fragment_types], epoch=30, verbose=1)

print(f"Prediction after training on fragment types: {requested_fragment_types}") 
prediction_precursor_df = psm_df.copy()
prediction_precursor_df.drop(columns=["frag_start_idx", "frag_stop_idx"], inplace=True)
predictions = model_interface.predict(prediction_precursor_df) # predict wil set the frag_start_idx and frag_stop_idx inplace
similarity_df = calculate_similarity(psm_df, prediction_precursor_df, frag_df.loc[:,supported_fragment_types], predictions)
print(f"Median similarity: {similarity_df['similarity'].median()}"  )

#save model
model_interface.save("/d0/tp/aiproteomics/tmp/tmp-ms2.pth")


c:\Users\letie\anaconda3\envs\peptdeep_env\lib\site-packages\alphabase\psm_reader\psm_reader.py:173: FutureWarning: Passed unknown arguments to LibraryReaderBase (fixed_C57=False) will be forbidden in alphabase>1.5.0.
  warnings.warn(
100%|██████████| 1/1 [00:00<00:00, 270.18it/s]
D:\DACN\dbl_alphapepdeep\alphapeptdeep\peptdeep\model\ms2.py:416: UserWarning: mask_modloss is deprecated and will be removed in the future. To mask the modloss fragments, the charged_frag_types should not include the modloss fragments.
  warnings.warn(


AttributeError: 'ModelManager' object has no attribute 'model'

In [ ]:
import pandas as pd

a = pd.read_csv("~/aiproteomics/tmp/1.txt", sep="\t")


In [ ]:
a

,Raw.file,Scan.number,Scan.index,Modified.sequence,Score,Charge,m.z,Retention.time,Matches,Intensities,Masses,Search.ID
0,QE2_130306_OPL1013_8x3_CRC_TiOx_rerun_CACO2_2,13473,11452,_AEEDEILNRS(UniMod:21)PR_,78.326,3,503.56288,40.171,y2;y5;y6;y7;y8;y9;y10;y11;y8-H2O;y9-H2O;y10-H2...,8208113;392600.9;991163.3;786324.5;1342676;222...,272.171396471234;709.31130202926;822.398938755...,1004
1,QE2_130306_OPL1013_8x3_CRC_TiOx_rerun_COLO205_2,12587,10207,_AEEDEILNRS(UniMod:21)PR_,55.011,3,503.56288,39.731,y2;y6;y8;y9;y10;y11;y11-NH3;y6*;y6-NH3;y7*;y8*...,1.646677E+07;3416965;2787151;4340183;1.034908E...,272.170905022262;822.398931634571;1064.5243490...,1004
2,QE2_130306_OPL1013_8x3_CRC_TiOx_rerun_DLD1_2,13341,11145,_AEEDEILNRS(UniMod:21)PR_,62.357,3,503.56288,39.897,y2;y6;y8;y9;y5*;y5-NH3;y6*;y6-NH3;y8*;y9*;y9-N...,8431017;1614818;1029005;1194362;427076.1;59139...,272.171340272003;822.399820171586;1064.5328842...,1004
3,QE2_130306_OPL1013_8x3_CRC_TiOx_rerun_HCT116_2,11908,9521,_AEEDEILNRS(UniMod:21)PR_,48.899,3,503.56288,39.606,y2;y6;y7;y9;y10;y11;y7-NH3;y9-H2O;y11-NH3;y7*;...,8874589;1112488;1027119;2727959;5845389;151896...,272.171088867789;822.397334264859;935.48271723...,1004
4,QE2_130306_OPL1013_8x3_CRC_TiOx_rerun_HT29_2_1...,11759,9317,_AEEDEILNRS(UniMod:21)PR_,46.460,2,754.84068,38.897,y2;y5;y6;y7;y8;y9;y10;y5*;y5-NH3;y7*;y8*;y9*;y...,4220644;406179.8;407346;206201.8;295958.1;3191...,272.170911276589;709.310814231477;822.39449710...,1004
...,...,...,...,...,...,...,...,...,...,...,...,...
47534,QE2_150602_OPL1005_EG_TiOx_BRCA_mouse_95,9547,6871,_AEEDEILNRS(UniMod:21)PR_,95.822,2,754.84068,36.734,y2;y3;y4;y5;y6;y8;y10;y3-H2O;y2-NH3;y3*;y5*;y6...,584979.4;5573.9;6888.3;12603.3;33249;58468.5;3...,272.171276277079;439.170912039902;595.28028188...,9
47535,QE2_150602_OPL1005_EG_TiOx_BRCA_mouse_95,9787,7089,_AEEDEILNRS(UniMod:21)PR_,176.890,2,754.84068,37.251,y2;y4;y5;y6;y7;y8;y10;y2-NH3;y6-NH3;y3*;y5*;y6...,1076814.6;36169;58953.2;75252;9866.6;93099.2;3...,272.17122220775;595.279381964272;709.312124925...,9
47536,QE2_150602_OPL1005_EG_TiOx_BRCA_mouse_95,9743,7049,_AEEDEILNRS(UniMod:21)PR_,172.570,3,503.56288,37.156,y2;y3;y5;y6;y7;y8;y9;y10;y11;y3-H2O;y8-H2O;y10...,10966129.5;189248.3;715060.6;1990628.9;931604....,272.171525441888;439.170055142226;709.31002082...,9
47537,QE2_150602_OPL1005_EG_TiOx_BRCA_mouse_96,9691,7050,_AEEDEILNRS(UniMod:21)PR_,125.220,2,754.84068,36.927,y2;y3;y5;y6;y7;y8;y2-NH3;y3*;y5*;y6*;y7*;y8*;y...,645009.4;14347.9;34290.7;33477.2;7278.8;65152....,272.17114876659;439.171284731879;709.312402530...,9
